# Stock Market Analysis — Next-Day Direction Prediction

End-to-end walkthrough: data, feature engineering, model evaluation, backtesting,
and artifact export for the Streamlit app.

**The two decisions that shape this notebook:**

1. **No shuffled cross-validation.** Financial time series break the i.i.d. assumption
   behind k-fold CV — shuffling puts tomorrow in the training set and yesterday in the
   test set. Everything here uses expanding-window walk-forward validation.
2. **Accuracy is not profit.** A model can be right 55% of the time and lose money if
   it's wrong on the days that move most, or if trading costs eat the edge. Every model
   is scored as a trading signal against buy-and-hold, net of costs.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (12, 5)

from src.data.fetch import fetch_prices, synthetic_ohlcv
from src.features.build_features import build_features, feature_columns
from src.models.train import compare_models, walk_forward_evaluate, MODELS
from src.backtest.engine import backtest, score_returns

## 1. Load data

Set `TICKER` and run. `OFFLINE = True` uses a synthetic random walk instead — useful
for testing the pipeline without network access, and as a negative control later.

In [ ]:
TICKER = "AAPL"
START = "2015-01-01"
OFFLINE = False

if OFFLINE:
    prices = synthetic_ohlcv()
    label = "SYNTHETIC"
else:
    prices = fetch_prices(TICKER, start=START)
    label = TICKER

print(f"{label}: {len(prices)} rows, {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, height_ratios=[3, 1], figsize=(12, 7))
ax1.plot(prices.index, prices["close"], lw=1, label="Close")
ax1.plot(prices.index, prices["close"].rolling(50).mean(), lw=1, label="SMA 50")
ax1.plot(prices.index, prices["close"].rolling(200).mean(), lw=1, label="SMA 200")
ax1.set_ylabel("Price"); ax1.legend(); ax1.set_title(f"{label} price history")
ax2.bar(prices.index, prices["volume"], width=1, color="lightslategray")
ax2.set_ylabel("Volume")
plt.tight_layout()

## 2. Return distribution

Worth looking at before modelling. Daily equity returns are roughly symmetric and
fat-tailed — the direction split is close to 50/50, which is why the majority-class
baseline is a genuinely hard benchmark.

In [ ]:
returns = prices["close"].pct_change().dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(returns, bins=80, color="steelblue", edgecolor="white")
ax1.axvline(0, color="black", lw=1, ls="--")
ax1.set_title("Daily return distribution"); ax1.set_xlabel("Return")
ax2.plot(returns.index, returns.rolling(21).std() * np.sqrt(252), lw=1, color="darkorange")
ax2.set_title("Rolling 21-day annualised volatility")
plt.tight_layout()

up_days = (returns > 0).mean()
print(f"Up days: {up_days:.2%}  |  Down days: {1 - up_days:.2%}")
print(f"Annualised vol: {returns.std() * np.sqrt(252):.2%}")
print(f"Skew: {returns.skew():.3f}  |  Excess kurtosis: {returns.kurtosis():.3f}")

## 3. Feature engineering

24 features across six groups. Every one is computed from information available at or
before the close of day *t*; the target is the return realised on day *t+1*.

| Group | Features |
| --- | --- |
| Momentum | Lagged returns (1, 2, 3, 5, 10d), 5d and 21d cumulative |
| Trend | Close vs SMA (5, 10, 20, 50), EMA 12/26 ratio |
| Volatility | 10d and 21d realised vol, vol ratio, high–low range |
| Oscillators | RSI(14), MACD, MACD histogram, Bollinger position |
| Volume | Volume vs 20d SMA, volume change |
| Calendar | Day of week, month |

In [ ]:
features = build_features(prices)
cols = feature_columns(features)

print(f"{len(features)} rows after warm-up, {len(cols)} features")
print(f"Target balance: {features['target_direction'].mean():.2%} up days")
features[cols].tail()

### Leakage check

The single most important cell in this notebook. If any feature used forward-looking
information, its value on a given date would change once later rows were removed.
Truncating the history and recomputing must produce identical values on the overlap.

In [ ]:
truncated = build_features(prices.iloc[:-100])
shared = truncated.index.intersection(features.index)

identical = np.allclose(
    features.loc[shared, cols].to_numpy(),
    truncated.loc[shared, cols].to_numpy(),
    rtol=1e-9,
)
print(f"Overlapping rows checked: {len(shared)}")
print(f"No lookahead leakage: {identical}")
assert identical, "FEATURE LEAKAGE DETECTED — do not trust anything below this cell"

corr = features[cols].corrwith(features["target_return"]).sort_values(key=abs, ascending=False)
print("\nTop feature correlations with next-day return:")
print(corr.head(8).to_string())

Those correlations are tiny — typically under 0.05. That is the honest starting
point for this problem, and it sets expectations for what follows.

## 4. Walk-forward model evaluation

`TimeSeriesSplit` builds expanding windows: fold 1 trains on the earliest slice and
tests on what follows, fold 2 trains on everything through fold 1's test set, and so on.
No model ever sees data from its own test period.

In [ ]:
leaderboard = compare_models(features, n_splits=5)
leaderboard

In [ ]:
baseline = leaderboard.loc[leaderboard["model"] == "baseline_majority", "mean_accuracy"].iloc[0]
best = leaderboard.iloc[0]

print(f"Majority-class baseline: {baseline:.4f}")
print(f"Best model ({best['model']}): {best['mean_accuracy']:.4f}")
print(f"Edge over baseline: {best['mean_accuracy'] - baseline:+.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(leaderboard["model"], leaderboard["mean_accuracy"],
        xerr=leaderboard["std_accuracy"], color="steelblue")
ax.axvline(baseline, color="crimson", ls="--", lw=1.5, label="Majority baseline")
ax.axvline(0.5, color="grey", ls=":", lw=1, label="Coin flip")
ax.set_xlim(0.4, 0.6); ax.set_xlabel("Walk-forward accuracy"); ax.legend()
plt.tight_layout()

In [ ]:
result = walk_forward_evaluate(features, model_name="gradient_boosting", n_splits=5)

fold_df = pd.DataFrame([{
    "fold": f.fold, "train_size": f.train_size, "test_size": f.test_size,
    "accuracy": f.accuracy, "f1": f.f1, "roc_auc": f.roc_auc,
} for f in result.folds])
print("Fold-by-fold — note the expanding training window:")
fold_df

Fold-to-fold variance is usually larger than the gap between models. That is a
signal in itself: it means apparent differences between models on this data are mostly
noise, and picking a winner on mean accuracy alone would be overfitting the evaluation.

## 5. Backtest

Converts the walk-forward predictions into a position (long when the model predicts up,
flat otherwise), charges transaction costs on every position change, and compares the
result against buy-and-hold over the same window.

In [ ]:
bt = backtest(features, model_name="gradient_boosting", n_splits=5, cost_bps=5.0)
strategy, benchmark = bt["strategy"], bt["buy_and_hold"]

pd.DataFrame({"strategy": strategy.summary(), "buy_and_hold": benchmark.summary()})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(strategy.equity_curve.index, strategy.equity_curve, lw=1.5, label="Model signal")
ax.plot(benchmark.equity_curve.index, benchmark.equity_curve, lw=1.5, label="Buy and hold")
ax.axhline(1.0, color="grey", ls=":", lw=1)
ax.set_ylabel("Growth of $1"); ax.set_title("Out-of-sample equity curve (5 bps costs)")
ax.legend(); plt.tight_layout()

print(f"Time in market: {bt['days_in_market']:.1%}")
print(f"Excess return vs buy-and-hold: {bt['excess_return']:+.2%}")

### Cost sensitivity

How fast does the edge disappear as trading gets more expensive? If the strategy only
works at zero cost, it doesn't work.

In [ ]:
rows = []
for cost in [0, 2, 5, 10, 20, 50]:
    r = backtest(features, model_name="gradient_boosting", n_splits=5, cost_bps=cost)
    rows.append({
        "cost_bps": cost,
        "total_return": r["strategy"].total_return,
        "sharpe": r["strategy"].sharpe_ratio,
        "excess_vs_hold": r["excess_return"],
    })

sensitivity = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sensitivity["cost_bps"], sensitivity["total_return"], marker="o", label="Strategy")
ax.axhline(benchmark.total_return, color="crimson", ls="--", label="Buy and hold")
ax.set_xlabel("Round-trip cost (bps)"); ax.set_ylabel("Total return"); ax.legend()
plt.tight_layout()
sensitivity

## 6. Negative control

Re-run the whole pipeline on a synthetic random walk. There is no signal to find, so
every model should land at the baseline. If a model appears to beat it here, the
pipeline is leaking — this is the check that makes the results above trustworthy.

In [ ]:
control = build_features(synthetic_ohlcv(n_days=1500, seed=99))
control_board = compare_models(control, n_splits=5)

control_baseline = control_board.loc[
    control_board["model"] == "baseline_majority", "mean_accuracy"].iloc[0]
control_best = control_board["mean_accuracy"].max()

print(control_board.to_string(index=False))
print(f"\nBaseline: {control_baseline:.4f}  |  Best: {control_best:.4f}")
print(f"Gap: {control_best - control_baseline:+.4f}  (should be near zero)")

## 7. Export artifacts

Refit on the full history and persist the model, scaler, and feature metadata for the
Streamlit app. The metadata file is the contract between training and serving: if the
app builds features in a different order than the model was trained on, predictions are
silently wrong, so the order is stored and verified at load time.

In [ ]:
from train_model import train

metadata = train(ticker=TICKER, start=START, model_name="gradient_boosting",
                 n_splits=5, offline=OFFLINE)

print("\nArtifacts written:")
print("  stock_model.joblib       fitted classifier")
print("  scaler.joblib            fitted StandardScaler")
print("  feature_metadata.joblib  feature order + training metrics")
print("\nLaunch the app with:  streamlit run app.py")

## Conclusions

**What the pipeline does.** Ingests OHLCV data with a Parquet cache layer, engineers 24
lag-safe time-series features, evaluates four models under expanding-window walk-forward
validation, and scores the resulting signal as a cost-aware trading strategy against
buy-and-hold.

**What the results show.** Accuracy sits marginally above the majority-class baseline,
and the edge usually does not survive realistic transaction costs. This is the expected
outcome — daily direction prediction from price history alone sits close to the
efficient-market limit.

**Why that's reported rather than tuned away.** It would be easy to produce an impressive
number here: shuffle the cross-validation folds, drop transaction costs, or tune
hyperparameters against the test set. All three are common in published stock-prediction
projects and all three are wrong. The value of this project is the evaluation
discipline — the leakage guard, the baseline comparison, the negative control, and the
cost sensitivity analysis — not a profitable strategy.

**Where a real edge would come from.** Not from more model complexity on the same
inputs. Alternative data — order-flow imbalance, options positioning, earnings-call
sentiment, cross-asset signals — is where the information actually lives.

*Not financial advice. Educational project only.*